# Notebook 03 - Feature Engineering and Preprocessing
### LendingClub Loan Default Prediction

**Author:** Talia Low | **Date:** March 2026

This notebook transforms the `train`, `val`, and `holdout` base tables produced by Notebook 02 into model-ready feature matrices.

**Objective:**
1. Load the pre-defined `train`, `val`, and `holdout` datasets,
2. Engineer additional features consistently across all splits,
3. Apply feature dropping and categorical cleaning consistently across all splits,
4. Fit the preprocessing pipeline on `df_train` only,
5. Transform `df_val` and `df_holdout` using the frozen training-fitted pipeline,
6. Save the processed matrices and preprocessing artefacts for downstream modelling.

**Important rule:**
All train-fitted preprocessing steps, including target encoding, missingness-based feature decisions, imputation, scaling, and categorical encoding, must be learned from `df_train` only.

# 1. Setup

In [1]:
# Imports
import gc
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)

In [2]:
# Paths
PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COL = "target"

## 2. Load and split inputs

### 2.1 Load Train, Validation, and Holdout Sets from Notebook 02

In [3]:
df_train = pd.read_csv(PROCESSED_DIR / "train.csv", low_memory=False)
df_val = pd.read_csv(PROCESSED_DIR / "val.csv", low_memory=False)
df_holdout = pd.read_csv(PROCESSED_DIR / "holdout.csv", low_memory=False)

print(f"Train set:      {df_train.shape[0]:,} rows x {df_train.shape[1]:,} columns")
print(f"Validation set: {df_val.shape[0]:,} rows x {df_val.shape[1]:,} columns")
print(f"Holdout set:    {df_holdout.shape[0]:,} rows x {df_holdout.shape[1]:,} columns")

print(f"\nTrain default rate:      {df_train[TARGET_COL].mean():.2%}")
print(f"Validation default rate: {df_val[TARGET_COL].mean():.2%}")
print(f"Holdout default rate:    {df_holdout[TARGET_COL].mean():.2%}")

Train set:      960,780 rows x 85 columns
Validation set: 240,195 rows x 85 columns
Holdout set:    69,339 rows x 85 columns

Train default rate:      19.93%
Validation default rate: 19.93%
Holdout default rate:    22.20%


## 2.2 Separate target from predictors 

In [4]:
# Separate target
y_train = df_train[TARGET_COL].values.copy()
y_val = df_val[TARGET_COL].values.copy()
y_holdout = df_holdout[TARGET_COL].values.copy()

df_train = df_train.drop(columns=[TARGET_COL]).copy()
df_val = df_val.drop(columns=[TARGET_COL]).copy()
df_holdout = df_holdout.drop(columns=[TARGET_COL]).copy()

print(f"y_train default rate:   {y_train.mean():.4f}")
print(f"y_val default rate:     {y_val.mean():.4f}")
print(f"y_holdout default rate: {y_holdout.mean():.4f}")

y_train default rate:   0.1993
y_val default rate:     0.1993
y_holdout default rate: 0.2220


## 3. Feature Selection and Leakage Review

Before building the preprocessing pipeline, we apply deterministic feature-selection and schema-cleaning decisions documented in Notebooks 01 and 02.

These include:
1. Dropping columns that should not be used as final raw predictors,
2. Applying multicollinearity-based feature reduction from Notebook 02,
3. Retaining helper raw fields such as `issue_d` and `earliest_cr_line` only until feature engineering is complete.

All such transformations must be applied consistently across `df_train`, `df_val`, and `df_holdout`.

### 3.1 Remove unusable columns

Before building the pipeline, we need to:
1. **Drop columns not usable at prediction time** - `issue_d` is the origination date used for splitting, not a predictor
2. **Drop addr_state identifiers** since it contains the same information as `zip3` 
3. **Drop near-constant features** - `disbursement_method` is >99% "Cash" - no predictive value

**Assumptions documented:**
- `earliest_cr_line` is converted to a numeric `credit_age_months` feature before dropping.
- A production model might use geographic features via target encoding, but for this portfolio project we prioritize a clean, interpretable pipeline.

In [5]:
# Columns to drop after feature engineering / schema consolidation

# Original exclusions
BASE_DROP_COLS = [
    "issue_d",              # Helper raw field used for feature engineering, not a final predictor
    "addr_state",           # Excluded for simplicity; zip3 retained as the geographic feature
    "disbursement_method",  # Near-constant (>99% Cash)
]

# Multicollinearity-based drops documented in Notebook 02
MULTICOLL_DROP = [
    # Cluster 1: Account counts — keep open_acc, num_actv_rev_tl, total_acc
    "num_sats",
    "num_rev_tl_bal_gt_0",
    "num_op_rev_tl",
    "num_rev_accts",
    "num_actv_bc_tl",
    "num_bc_sats",
    "num_bc_tl",

    # Cluster 2: Recent activity — keep acc_open_past_24mths, num_tl_op_past_12m
    "open_rv_24m",
    "open_acc_6m",
    "open_rv_12m",

    # Cluster 3: Credit limits — keep revol_bal, total_rev_hi_lim
    "bc_open_to_buy",
    "total_bc_limit",

    # Cluster 4: Delinquency timing — keep mths_since_last_delinq, mths_since_last_major_derog
    "mths_since_recent_bc_dlq",
    "mths_since_recent_revol_delinq",

    # Cluster 5: Balance aggregates — keep tot_cur_bal
    "tot_hi_cred_lim",
    "avg_cur_bal",

    # Cluster 6: Utilization — keep revol_util
    "bc_util",
    "percent_bc_gt_75",

    # Cluster 7: Installment balances — keep total_bal_ex_mort
    "total_bal_il",
    "total_il_high_credit_limit",

    # Cluster 8: Current delinquency — keep acc_now_delinq
    "num_tl_30dpd",

    # Cluster 10: Recent installment — drop both
    "open_il_24m",
    "open_il_12m",

    # Cluster 11: Public records — keep pub_rec
    "tax_liens",
]

ALL_DROP_COLS = BASE_DROP_COLS + MULTICOLL_DROP

existing_drop = [c for c in ALL_DROP_COLS if c in df_train.columns]
missing_drop = [c for c in ALL_DROP_COLS if c not in df_train.columns]

print(f"Dropping {len(existing_drop)} columns ({len(missing_drop)} not found in train data)")
print(f"  Base exclusions found: {len([c for c in BASE_DROP_COLS if c in df_train.columns])}")
print(f"  Multicollinearity drops found: {len([c for c in MULTICOLL_DROP if c in df_train.columns])}")
print(f"  Remaining features after drop: {df_train.shape[1] - len(existing_drop)}")

Dropping 27 columns (0 not found in train data)
  Base exclusions found: 3
  Multicollinearity drops found: 24
  Remaining features after drop: 57


## 4. Feature engineering

Based on Notebook 02 findings, we engineer the following derived features:
| New Feature | Formula | Intuition |
|---|---|---|
| `credit_age_months` | `issue_d - earliest_cr_line` | Credit history length at origination |
| `loan_to_income` | `loan_amnt / annual_inc` | Loan burden relative to income |
| `installment_to_income` | `installment * 12 / annual_inc` | Payment burden relative to annual income |
| `revol_bal_to_income` | `revol_bal / annual_inc` | Revolving debt burden relative to income |
| `open_acc_ratio` | `open_acc / total_acc` | Share of accounts currently open |
| `revol_bal_to_limit` | `revol_bal / total_rev_hi_lim` | Revolving balance relative to revolving limit |
| `total_debt_to_income` | `total_bal_ex_mort / annual_inc` | Total non-mortgage debt burden relative to income |
| `bc_open_to_buy_ratio` | `bc_open_to_buy / total_bc_limit` | Available unused bankcard capacity |
| `avg_bal_per_account` | `tot_cur_bal / open_acc` | Average balance per open account |
| `delinq_ratio` | `num_accts_ever_120_pd / total_acc` | Historical severe delinquency rate |
| `recent_inquiry_intensity` | `inq_last_6mths / (open_acc + 1)` | Recent inquiry pressure relative to account base |
| `mort_acc_ratio` | `mort_acc / total_acc` | Share of accounts that are mortgage-related |
| `log_annual_inc` | `log1p(annual_inc)` | Reduces right skew |
| `log_revol_bal` | `log1p(revol_bal)` | Reduces right skew |
| `log_loan_amnt` | `log1p(loan_amnt)` | Reduces right skew |
| `subgrade_dti_interaction` | `subgrade_num * dti` | Combines internal grade and debt burden |
| `missing_count` | row-wise missing count | Captures overall file incompleteness |

These transformations are applied consistently across `df_train`, `df_val`, and `df_holdout`

In [6]:
def engineer_features(df):
    df = df.copy()

    # Credit age at origination
    if {"issue_d", "earliest_cr_line"}.issubset(df.columns):
        issue_dt = pd.to_datetime(df["issue_d"], errors="coerce")
        ecl = pd.to_datetime(df["earliest_cr_line"], errors="coerce")
        df["credit_age_months"] = ((issue_dt - ecl).dt.days / 30.44).round(0)

    # Safe denominators
    safe_inc = df["annual_inc"].replace(0, np.nan) if "annual_inc" in df.columns else np.nan
    safe_total_acc = df["total_acc"].replace(0, np.nan) if "total_acc" in df.columns else np.nan
    safe_open_acc = df["open_acc"].replace(0, np.nan) if "open_acc" in df.columns else np.nan
    safe_rev_limit = df["total_rev_hi_lim"].replace(0, np.nan) if "total_rev_hi_lim" in df.columns else np.nan
    safe_bc_limit = df["total_bc_limit"].replace(0, np.nan) if "total_bc_limit" in df.columns else np.nan

    # Existing features
    if {"loan_amnt", "annual_inc"}.issubset(df.columns):
        df["loan_to_income"] = df["loan_amnt"] / safe_inc

    if {"installment", "annual_inc"}.issubset(df.columns):
        df["installment_to_income"] = (df["installment"] * 12) / safe_inc

    if {"revol_bal", "annual_inc"}.issubset(df.columns):
        df["revol_bal_to_income"] = df["revol_bal"] / safe_inc

    # New domain features
    if {"open_acc", "total_acc"}.issubset(df.columns):
        df["open_acc_ratio"] = df["open_acc"] / safe_total_acc

    if {"revol_bal", "total_rev_hi_lim"}.issubset(df.columns):
        df["revol_bal_to_limit"] = df["revol_bal"] / safe_rev_limit

    if {"total_bal_ex_mort", "annual_inc"}.issubset(df.columns):
        df["total_debt_to_income"] = df["total_bal_ex_mort"] / safe_inc

    if {"bc_open_to_buy", "total_bc_limit"}.issubset(df.columns):
        df["bc_open_to_buy_ratio"] = df["bc_open_to_buy"] / safe_bc_limit

    if {"tot_cur_bal", "open_acc"}.issubset(df.columns):
        df["avg_bal_per_account"] = df["tot_cur_bal"] / safe_open_acc

    if {"num_accts_ever_120_pd", "total_acc"}.issubset(df.columns):
        df["delinq_ratio"] = df["num_accts_ever_120_pd"] / safe_total_acc

    if {"inq_last_6mths", "open_acc"}.issubset(df.columns):
        df["recent_inquiry_intensity"] = df["inq_last_6mths"] / (df["open_acc"] + 1)

    if {"mort_acc", "total_acc"}.issubset(df.columns):
        df["mort_acc_ratio"] = df["mort_acc"] / safe_total_acc

    # Log transforms
    if "annual_inc" in df.columns:
        df["log_annual_inc"] = np.log1p(df["annual_inc"].clip(lower=0))

    if "revol_bal" in df.columns:
        df["log_revol_bal"] = np.log1p(df["revol_bal"].clip(lower=0))

    if "loan_amnt" in df.columns:
        df["log_loan_amnt"] = np.log1p(df["loan_amnt"].clip(lower=0))

    # Interaction: sub_grade x dti
    if {"sub_grade", "dti"}.issubset(df.columns):
        subgrade_order = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]
        subgrade_map = {g: i + 1 for i, g in enumerate(subgrade_order)}
        df["subgrade_num"] = df["sub_grade"].map(subgrade_map)
        df["subgrade_dti_interaction"] = df["subgrade_num"] * df["dti"]

    # Missing count
    df["missing_count"] = df.isna().sum(axis=1)

    # Drop helper raw fields
    drop_after_engineering = ["issue_d", "earliest_cr_line"]
    df = df.drop(columns=[c for c in drop_after_engineering if c in df.columns])

    return df

In [7]:
df_train = engineer_features(df_train)
df_val = engineer_features(df_val)
df_holdout = engineer_features(df_holdout)

# Drop base exclusions + multicollinearity-based drops consistently across all splits
df_train = df_train.drop(columns=[c for c in existing_drop if c in df_train.columns]).copy()
df_val = df_val.drop(columns=[c for c in existing_drop if c in df_val.columns]).copy()
df_holdout = df_holdout.drop(columns=[c for c in existing_drop if c in df_holdout.columns]).copy()

print(f"Training features after engineering/drop:   {df_train.shape[1]} columns")
print(f"Validation features after engineering/drop: {df_val.shape[1]} columns")
print(f"Holdout features after engineering/drop:    {df_holdout.shape[1]} columns")

/var/folders/k0/3dmr3r512xv0z12cwyxd_3cc0000gn/T/ipykernel_14495/108498645.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_dt = pd.to_datetime(df["issue_d"], errors="coerce")
/var/folders/k0/3dmr3r512xv0z12cwyxd_3cc0000gn/T/ipykernel_14495/108498645.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ecl = pd.to_datetime(df["earliest_cr_line"], errors="coerce")
/var/folders/k0/3dmr3r512xv0z12cwyxd_3cc0000gn/T/ipykernel_14495/108498645.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  issue_dt = pd.to_datetime(df["issue_d"], errors="coerce")
/var/folders/k0/3dmr3

Training features after engineering/drop:   74 columns
Validation features after engineering/drop: 74 columns
Holdout features after engineering/drop:    74 columns


## 5. Identify Feature Types and Missingness

### 5.1 Identify numeric and categorical columns

In [8]:
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

# zip3 is numeric (float) but is a high-cardinality categorical - handle separately
zip3_col = "zip3"
if zip3_col in numeric_cols:
    numeric_cols.remove(zip3_col)

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"High-cardinality feature (target-encoded): {zip3_col}")

Numeric features: 66
Categorical features (7): ['term', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status']
High-cardinality feature (target-encoded): zip3


In [9]:
# Inspect categorical features
for col in cat_cols:
    print(f"\n{col} ({df_train[col].nunique()} unique):")
    print(df_train[col].value_counts().head(6))


term (2 unique):
term
36 months    732218
60 months    228562
Name: count, dtype: int64

sub_grade (35 unique):
sub_grade
C1    61355
B4    60635
B3    59754
B5    58932
C2    56920
C3    53614
Name: count, dtype: int64

emp_length (11 unique):
emp_length
10+ years    316628
2 years       87267
3 years       77011
< 1 year      74990
1 year        63380
5 years       60129
Name: count, dtype: int64

home_ownership (6 unique):
home_ownership
MORTGAGE    473032
RENT        386100
OWN         101299
ANY            199
OTHER          112
NONE            38
Name: count, dtype: int64

verification_status (3 unique):
verification_status
Source Verified    373219
Verified           304537
Not Verified       283024
Name: count, dtype: int64

purpose (14 unique):
purpose
debt_consolidation    561269
credit_card           213989
home_improvement       60903
other                  52481
major_purchase         20303
small_business         11222
Name: count, dtype: int64

initial_list_status (2 uni

### 5.2 Missing-value strategy

EDA (Notebook 02) showed that missingness is **informative** for several features - the default rate differs meaningfully between rows where a value is present vs missing. This suggests Missing Not At Random (MNAR) patterns, likely because certain bureau fields are only populated for borrowers with specific credit histories.

**Strategy:**
1. For numeric features with >5% missingness: create a binary `_missing` indicator flag,
   then impute the original column with the training-set median.
2. For numeric features with ≤5% missingness: impute with the training-set median only.
3. For categorical features: `emp_length` missing values are filled with "Unknown" (treated
   as a separate category); other categoricals use most-frequent imputation.

In [10]:
# Identify high-missingness numeric features (based on TRAINING subset only)
high_missing = [c for c in numeric_cols if df_train[c].isna().mean() > 0.05]

# Proportion of missing values
print(f"Features with >5% missing ({len(high_missing)}):")
for c in high_missing:
    print(f"  {c}: {df_train[c].isna().mean():.1%}")

Features with >5% missing (29):
  mths_since_last_delinq: 50.2%
  mths_since_last_record: 83.0%
  mths_since_last_major_derog: 73.7%
  tot_coll_amt: 5.6%
  tot_cur_bal: 5.6%
  open_act_il: 67.2%
  mths_since_rcnt_il: 68.1%
  il_util: 71.5%
  max_bal_bc: 67.2%
  all_util: 67.2%
  total_rev_hi_lim: 5.6%
  inq_fi: 67.2%
  total_cu_tl: 67.2%
  inq_last_12m: 67.2%
  mo_sin_old_il_acct: 8.4%
  mo_sin_old_rev_tl_op: 5.6%
  mo_sin_rcnt_rev_tl_op: 5.6%
  mo_sin_rcnt_tl: 5.6%
  mths_since_recent_inq: 13.4%
  num_accts_ever_120_pd: 5.6%
  num_actv_rev_tl: 5.6%
  num_il_tl: 5.6%
  num_tl_120dpd_2m: 9.3%
  num_tl_90g_dpd_24m: 5.6%
  num_tl_op_past_12m: 5.6%
  pct_tl_nvr_dlq: 5.6%
  revol_bal_to_limit: 5.7%
  avg_bal_per_account: 5.6%
  delinq_ratio: 5.6%


In [11]:
# Create a new column and add missing indicator flags
for col in high_missing:
    flag_name = f"{col}_missing"
    # For each dataset, create a 0/1 flag
    df_train[flag_name] = df_train[col].isna().astype(np.int8)
    df_val[flag_name] = df_val[col].isna().astype(np.int8)
    df_holdout[flag_name] = df_holdout[col].isna().astype(np.int8)

missing_flag_cols = [f"{c}_missing" for c in high_missing]
numeric_cols_extended = numeric_cols + missing_flag_cols

print(f"\nExtended numeric features: {len(numeric_cols_extended)} "
      f"(original {len(numeric_cols)} + {len(missing_flag_cols)} indicators)")


Extended numeric features: 95 (original 66 + 29 indicators)


## 6. Categorical Handling

### 6.1 Categorical cleaning

In [12]:
def clean_categoricals(df):
    """Standardize categorical values before pipeline encoding"""
    df = df.copy()

    if "term" in df.columns:
        df["term"] = df["term"].str.strip()

    if "home_ownership" in df.columns:
        df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})

    if "emp_length" in df.columns:
        df["emp_length"] = df["emp_length"].fillna("Unknown")

    return df

df_train = clean_categoricals(df_train)
df_val = clean_categoricals(df_val)
df_holdout = clean_categoricals(df_holdout)

print("home_ownership values:", sorted(df_train["home_ownership"].unique()))
print("term values:", sorted(df_train["term"].unique()))
print("emp_length missing after cleaning:", df_train["emp_length"].isna().sum())

home_ownership values: ['MORTGAGE', 'OTHER', 'OWN', 'RENT']
term values: ['36 months', '60 months']
emp_length missing after cleaning: 0


### 6.2 Geographic handling for `zip3`

`zip3` is a high-cardinality geographic feature. To reduce the risk that its effect is driven mainly by encoding choice, we test two simpler alternatives:

- dropping `zip3` entirely
- out-of-fold target encoding with stronger smoothing

All mappings are learned from the training data only and then applied to validation and holdout.

In [13]:
# zip3 configuration
# ZIP3_STRATEGY controls how the raw zip3 feature is handled.
# Options:
#   - "drop"       : remove zip3 entirely
#   - "target_oof" : out-of-fold target encoding on train, then
#                    apply final mapping to val and holdout
ZIP3_STRATEGY = "target_oof"

# Smoothing strength for target encoding.
# Higher values shrink zip3 means more strongly toward the global mean.
ZIP3_SMOOTHING = 40

# Rare zip3 groups are collapsed into "RARE" before encoding.
# This avoids very small groups having unstable values.
ZIP3_MIN_COUNT = 50


class TargetEncoder:
    """
    Smoothed target encoder for one column.

    For each category:
        encoded_value = (count * group_mean + smoothing * global_mean) / (count + smoothing)

    This prevents tiny groups from getting extreme target-mean values.
    """

    def __init__(self, smoothing=100):
        self.smoothing = smoothing

    def fit(self, X, y):
        col = X.iloc[:, 0].astype("object")
        stats = pd.DataFrame({"val": col, "target": y})
        agg = stats.groupby("val")["target"].agg(["mean", "count"])

        self.global_mean_ = float(np.mean(y))

        smooth = (
            agg["count"] * agg["mean"] + self.smoothing * self.global_mean_
        ) / (agg["count"] + self.smoothing)

        self.mapping_ = smooth.to_dict()
        return self

    def transform(self, X):
        col = X.iloc[:, 0].astype("object")
        return col.map(self.mapping_).fillna(self.global_mean_).values


def bucket_rare_zip3(train_col, val_col, holdout_col, min_count=50):
    """
    Collapse rare zip3 values into 'RARE' based on TRAIN frequencies only.
    """
    counts = train_col.astype("object").value_counts(dropna=False)
    keepers = set(counts[counts >= min_count].index)

    def map_rare(s):
        s = s.astype("object").copy()
        s = s.where(s.isin(keepers), "RARE")
        s = s.fillna("MISSING")
        return s

    return map_rare(train_col), map_rare(val_col), map_rare(holdout_col)

In [14]:
# Start with a default value so the variable always exists.
# This prevents NameError later when saving preprocessing artifacts.
zip3_encoder = None

# Only run zip3 handling if the raw zip3 column is still present
if "zip3" in df_train.columns:

    # Collapse rare zip3 groups into "RARE" using TRAIN frequencies only
    z_train, z_val, z_holdout = bucket_rare_zip3(
        df_train["zip3"],
        df_val["zip3"],
        df_holdout["zip3"],
        min_count=ZIP3_MIN_COUNT,
    )

    # Option 1: drop zip3 entirely
    if ZIP3_STRATEGY == "drop":
        pass

    # Option 2: out-of-fold target encoding
    # Train rows are encoded using folds so that each row is encoded
    # by a model that did NOT see its own target value.
    # This reduces leakage / overfitting.
    elif ZIP3_STRATEGY == "target_oof":
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        # Placeholder for out-of-fold encoded values for the training set
        zip3_train_encoded = np.zeros(len(df_train), dtype=float)

        # Build OOF encodings fold by fold
        for tr_idx, oof_idx in skf.split(z_train.to_frame(), y_train):
            enc = TargetEncoder(smoothing=ZIP3_SMOOTHING)

            # Fit encoder on fold-train only
            enc.fit(z_train.iloc[tr_idx].to_frame(), y_train[tr_idx])

            # Apply encoder to fold-validation rows
            zip3_train_encoded[oof_idx] = enc.transform(
                z_train.iloc[oof_idx].to_frame()
            )

        # Fit one final encoder on the FULL training set
        # This final encoder is used for val and holdout
        final_enc = TargetEncoder(smoothing=ZIP3_SMOOTHING)
        final_enc.fit(z_train.to_frame(), y_train)

        # Save encoded columns
        df_train["zip3_encoded"] = zip3_train_encoded
        df_val["zip3_encoded"] = final_enc.transform(z_val.to_frame())
        df_holdout["zip3_encoded"] = final_enc.transform(z_holdout.to_frame())

        # Make sure the new encoded column is treated as numeric
        numeric_cols_extended.append("zip3_encoded")

        # Save the fitted final encoder as the artifact
        zip3_encoder = final_enc

    else:
        raise ValueError(f"Unknown ZIP3_STRATEGY: {ZIP3_STRATEGY}")

    # Once zip3 has been handled, remove the raw zip3 column
    # so it does not enter the model twice
    df_train.drop(columns=["zip3"], inplace=True)
    df_val.drop(columns=["zip3"], inplace=True)
    df_holdout.drop(columns=["zip3"], inplace=True)

# Print settings so it is clear which experiment was run
print("ZIP3_STRATEGY =", ZIP3_STRATEGY)
print("ZIP3_SMOOTHING =", ZIP3_SMOOTHING)
print("ZIP3_MIN_COUNT =", ZIP3_MIN_COUNT)
print("zip3_encoder created:", zip3_encoder is not None)

ZIP3_STRATEGY = target_oof
ZIP3_SMOOTHING = 40
ZIP3_MIN_COUNT = 50
zip3_encoder created: True


## 7. Preprocessing Pipeline

The sklearn `ColumnTransformer` handles the final train-fitted preprocessing steps:

| Feature Group | Transformer | Details |
|---|---|---|
| Numeric (extended) | `SimpleImputer(median)` -> `StandardScaler` | Median imputation and scaling |
| `term` | `OrdinalEncoder` | Natural ordering: 36 months < 60 months |
| `sub_grade` | `OrdinalEncoder` | Risk ordering: A1 (safest) -> G5 (riskiest) |
| `emp_length` | `OrdinalEncoder` | Employment duration ordering + `Unknown` |
| Other categoricals | `OneHotEncoder(drop="first")` | One-hot encoding with infrequent-category handling |

All components are fitted on `df_train` only.

### 7.1 Build pipeline

In [15]:
# Define the correct order for ordinal categorical variables

# sub_grade runs from A1 (best) to G5 (worst)
subgrade_order = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]

# loan term has a natural order: 36 months < 60 months
term_order = ["36 months", "60 months"]

# employment length also has a natural order
emp_length_order = [
    "< 1 year", "1 year", "2 years", "3 years", "4 years",
    "5 years", "6 years", "7 years", "8 years", "9 years",
    "10+ years", "Unknown",
]

# Columns that should be ordinal-encoded because their categories are ordered
ordinal_cols = ["term", "sub_grade", "emp_length"]

# All remaining categorical columns will be one-hot encoded instead
onehot_cols = [c for c in cat_cols if c not in ordinal_cols]

print(f"Ordinal-encoded: {ordinal_cols}")
print(f"One-hot-encoded: {onehot_cols}")

Ordinal-encoded: ['term', 'sub_grade', 'emp_length']
One-hot-encoded: ['home_ownership', 'verification_status', 'purpose', 'initial_list_status']


In [16]:
# 1. Define preprocessing pipeline for numeric features
# Step 1: fill missing numeric values with the median
# Step 2: standardise numeric features to mean=0, std=1
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 2. Define preprocessing pipeline for 'term'
# Fill missing values with the most common term
# Then map the ordered categories to numeric values
term_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[term_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])


# 3. Define preprocessing pipeline for 'sub_grade'
# Fill missing values with the most common sub-grade
# Then encode A1 -> A2 -> ... -> G5 in the specified order
subgrade_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[subgrade_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])


# 4. Define preprocessing pipeline for 'emp_length'
# Fill missing values explicitly as 'Unknown'
# Then encode employment length in the specified order
emp_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OrdinalEncoder(
        categories=[emp_length_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])


# 5. Define preprocessing pipeline for remaining categorical columns
# Remaining columns: 'home_ownership', 'verification_status', 'purpose', 'initial_list_status'
# Fill missing values with the most common category
# Then apply one-hot encoding
#
# drop="first":
#   drops one dummy column per feature to avoid redundancy
#
# handle_unknown="infrequent_if_exist", min_frequency=100:
#   rare or unseen categories are grouped more robustly
onehot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="first",
        sparse_output=False,
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    )),
])


# 6. Combine all sub-pipelines into one master preprocessor
# Each pipeline is applied only to its assigned columns
# ("name", transformer, columns)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols_extended),  # numeric features
        ("term", term_pipeline, ["term"]),                # ordered term feature
        ("subgrade", subgrade_pipeline, ["sub_grade"]),   # ordered credit grade
        ("emp", emp_pipeline, ["emp_length"]),            # ordered employment length
        ("onehot", onehot_pipeline, onehot_cols),         # all other categorical features
    ],
    remainder="drop",                 # drop any columns not explicitly listed
    verbose_feature_names_out=True,   # output feature names include transformer prefix
)

print("Pipeline defined successfully.")
print(f"Total input columns: {len(numeric_cols_extended) + len(ordinal_cols) + len(onehot_cols)}")

Pipeline defined successfully.
Total input columns: 103


### 7.2 Fit and transform

The preprocessing pipeline is fitted **only on `df_train`**.

The fitted pipeline is then used to transform:
- `df_train`
- `df_val`
- `df_holdout`

This preserves a clean separation between training and evaluation datasets.

In [17]:
# Fit on training subset only
preprocessor.fit(df_train)
print("Pipeline fitted on training subset.")

Pipeline fitted on training subset.


In [18]:
# Transform training set
X_train = preprocessor.transform(df_train).astype(np.float32)
print(f"Processed training set: {X_train.shape}")
gc.collect()

Processed training set: (960780, 118)


89

In [19]:
# Transform validation and holdout sets using the frozen, training-fitted pipeline
X_val = preprocessor.transform(df_val).astype(np.float32)
X_holdout = preprocessor.transform(df_holdout).astype(np.float32)

print(f"Processed validation set: {X_val.shape}")
print(f"Processed holdout set:    {X_holdout.shape}")
gc.collect()

Processed validation set: (240195, 118)
Processed holdout set:    (69339, 118)


133

In [20]:
# Get feature names from the fitted pipeline
feature_names = list(preprocessor.get_feature_names_out())
print(f"Total output features: {len(feature_names)}")
print(f"\nSample feature names:")
print(f"  Numeric: {feature_names[:5]} ...")
print(f"  Ordinal: {[f for f in feature_names if 'term' in f or 'subgrade' in f or 'emp__' in f]}")
print(f"  One-hot: {feature_names}")

Total output features: 118

Sample feature names:
  Numeric: ['num__loan_amnt', 'num__int_rate', 'num__installment', 'num__annual_inc', 'num__dti'] ...
  Ordinal: ['num__subgrade_num', 'num__subgrade_dti_interaction', 'term__term', 'subgrade__sub_grade', 'emp__emp_length']
  One-hot: ['num__loan_amnt', 'num__int_rate', 'num__installment', 'num__annual_inc', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__mths_since_last_delinq', 'num__mths_since_last_record', 'num__open_acc', 'num__pub_rec', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__collections_12_mths_ex_med', 'num__mths_since_last_major_derog', 'num__acc_now_delinq', 'num__tot_coll_amt', 'num__tot_cur_bal', 'num__open_act_il', 'num__mths_since_rcnt_il', 'num__il_util', 'num__max_bal_bc', 'num__all_util', 'num__total_rev_hi_lim', 'num__inq_fi', 'num__total_cu_tl', 'num__inq_last_12m', 'num__acc_open_past_24mths', 'num__chargeoff_within_12_mths', 'num__delinq_amnt', 'num__mo_sin_old_il_acct', 'num__mo_si

## 8. Quality Checks and Outputs

### 8.1 Quality checks

In [21]:
# Final quality check: look for any problematic numeric values after preprocessing
for name, arr in [("Train", X_train), ("Val", X_val), ("Holdout", X_holdout)]:
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    print(f"{name:8s} -- NaN: {n_nan}, Inf: {n_inf}")

Train    -- NaN: 0, Inf: 0
Val      -- NaN: 0, Inf: 0
Holdout  -- NaN: 0, Inf: 0


In [22]:
# Clean up any remaining Inf values (from division-derived features where income was 0)
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
X_holdout = np.nan_to_num(X_holdout, nan=0.0, posinf=0.0, neginf=0.0)

print("After cleanup:")
for name, arr in [("Train", X_train), ("Val", X_val), ("Holdout", X_holdout)]:
    print(f"  {name:8s} -- NaN: {np.isnan(arr).sum()}, Inf: {np.isinf(arr).sum()}")

After cleanup:
  Train    -- NaN: 0, Inf: 0
  Val      -- NaN: 0, Inf: 0
  Holdout  -- NaN: 0, Inf: 0


### 8.2 Feature summary

In [23]:
# Quick summary of processed features
summary_df = pd.DataFrame({
    "feature": feature_names,
    "train_mean": X_train.mean(axis=0).round(3),
    "train_std": X_train.std(axis=0).round(3),
    "val_mean": X_val.mean(axis=0).round(3),
    "holdout_mean": X_holdout.mean(axis=0).round(3),
})
zero_var = (summary_df["train_std"] == 0).sum()
print(f"Features with zero variance in training: {zero_var}")
summary_df.head(15)

Features with zero variance in training: 0


,feature,train_mean,train_std,val_mean,holdout_mean
0,num__loan_amnt,-0.0,0.999,-0.000,-0.074
1,num__int_rate,-0.0,1.000,-0.001,0.146
2,num__installment,0.0,1.000,-0.001,-0.064
3,num__annual_inc,0.0,1.000,0.002,0.065
4,num__dti,0.0,1.000,-0.002,-0.082
5,num__delinq_2yrs,0.0,0.998,0.004,-0.018
6,num__inq_last_6mths,0.0,1.002,-0.005,-0.059
7,num__mths_since_last_delinq,-0.0,0.998,-0.003,0.024
8,num__mths_since_last_record,0.0,1.000,0.000,0.112
9,num__open_acc,-0.0,0.999,-0.002,-0.020


### 8.3 Save outputs
We save:
- processed feature matrices (`X_train`, `X_val`, `X_holdout`) as NumPy arrays
- target vectors (`y_train`, `y_val`, `y_holdout`)
- feature names
- fitted preprocessing artefacts, including:
  - the column drop list
  - the fitted `zip3` target encoder
  - the fitted `ColumnTransformer`
  - the high-missing column list
  - the missing-indicator column list

These artefacts are sufficient to reproduce the fitted preprocessing logic, although they do not form a single end-to-end raw-input pipeline.

In [24]:
np.save(PROCESSED_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DIR / "X_val.npy", X_val)
np.save(PROCESSED_DIR / "X_holdout.npy", X_holdout)

np.save(PROCESSED_DIR / "y_train.npy", y_train)
np.save(PROCESSED_DIR / "y_val.npy", y_val)
np.save(PROCESSED_DIR / "y_holdout.npy", y_holdout)

with open(PROCESSED_DIR / "feature_names.pkl", "wb") as f:
    pickle.dump(feature_names, f)

# Save fitted preprocessing artefacts
preprocessing_artifacts = {
    "drop_cols": ALL_DROP_COLS,
    "zip3_encoder": zip3_encoder,
    "preprocessor": preprocessor,
    "feature_names": feature_names,
    "high_missing_cols": high_missing,
    "missing_flag_cols": missing_flag_cols,
}

with open(MODEL_DIR / "preprocessing_pipeline.pkl", "wb") as f:
    pickle.dump(preprocessing_artifacts, f)

print("All outputs saved successfully:")
print(f"  X_train:   {X_train.shape}")
print(f"  X_val:     {X_val.shape}")
print(f"  X_holdout: {X_holdout.shape}")
print(f"  y_train:   {y_train.shape}")
print(f"  y_val:     {y_val.shape}")
print(f"  y_holdout: {y_holdout.shape}")
print(f"  Features:  {len(feature_names)}")
print(f"\nSaved preprocessing artefacts to: {MODEL_DIR / 'preprocessing_pipeline.pkl'}")

All outputs saved successfully:
  X_train:   (960780, 118)
  X_val:     (240195, 118)
  X_holdout: (69339, 118)
  y_train:   (960780,)
  y_val:     (240195,)
  y_holdout: (69339,)
  Features:  118

Saved preprocessing artefacts to: ../models/preprocessing_pipeline.pkl


### 8.4 Preprocessing summary

| Step | Details |
|---|---|
| **Input splits** | `train.csv`, `val.csv`, and `holdout.csv` loaded from Notebook 02 |
| **Multicollinearity drops** | 24 redundant features removed based on Notebook 02 |
| **Other dropped columns** | `issue_d`, `earliest_cr_line`, `addr_state`, `disbursement_method` |
| **Engineered features** | `credit_age_months`, `loan_to_income`, `installment_to_income`, `revol_bal_to_income` |
| **`zip3` encoding** | Smoothed target encoding fitted on `df_train` only |
| **Missing indicators** | Binary flags for features with >5% missingness in `df_train` |
| **Numeric imputation** | Median imputation |
| **Numeric scaling** | StandardScaler |
| **Ordinal encoding** | `term`, `sub_grade`, `emp_length` |
| **One-hot encoding** | Remaining categorical predictors |
| **Leakage prevention** | All learned preprocessing fitted on `df_train` only |

**Pipeline fit on:** `df_train` only  
**Output arrays:** `X_train`, `X_val`, `X_holdout`, plus corresponding target vectors